# step back prompt

In [37]:
# %pip install mplfinance

In [38]:
import pandas as pd
import os

def read_taifex(path:str, product: str, product_m_or_w: str):
    '''
        讀取台灣期貨交易所的歷史資料，並選取特定商品代號及月份(週別)的資料

        年月日: str 
        商品代號: str
        商品月份: str

        回傳: DataFrame
    '''
    
    # 讀取 CSV 檔案，並將所有元素轉換成字串
    if os.path.exists(path):
        df = pd.read_csv(path, encoding="BIG5", dtype=str)
    else:
        return pd.DataFrame()
     
    # 移除所有元素的空白
    df = df.map(lambda x: x.strip())

    # 當 '到期月份(週別)' 包含 '/' 時，將 '成交價格' 的值改為 '近月價格' 的值
    df.loc[df['到期月份(週別)'].str.contains('/'), '成交價格'] = df['近月價格']
    
    # 到期月份(週別)中有些資料是202404/202405，代表轉倉，因此分割字串並選取第一個元素並存到新欄位 "商品月份"
    df['商品月份'] = df['到期月份(週別)'].str.split('/').str[0]

    df = df[df["商品代號"] == product]
    df = df[df["商品月份"] == product_m_or_w]

    # 將"交易日期" 和 "交易時間" 合併成一個新的欄位 "交易時間"
    df['Date'] = df['成交日期'] + df['成交時間']

    # 然後，將 "交易時間" 轉換成 datetime 對象
    # 假設 "交易日期" 是 "YYYYMMDD" 的格式，"交易時間" 是 "HHMMSS" 的格式
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d%H%M%S')

    df.set_index('Date', inplace=True)
    df = df.drop(columns=['成交日期'])
    df = df.drop(columns=['成交時間'])

    return df

In [39]:
import pandas as pd
import os
from datetime import datetime, time
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import mplfinance as mpf

# 假設你的 DataFrame 是 df
def to_kline(interval:str, df:pd.DataFrame):
    '''
    將原始交易資料轉換為 K 線資料

    參數:
        interval: K 線的時間間隔，例如 '5min' 代表 5 分鐘
        df: 原始交易資料，必須包含 '成交數量(B+S)' 和 '成交價格' 兩個欄位

    回傳值
        df_kline: K 線資料，包含 'Open', 'High', 'Low', 'Close', 'Volume' 五個欄位
    '''

    df.index = pd.to_datetime(df.index)  # 確保索引是 datetime 對象
    df['成交數量(B+S)'] = df['成交數量(B+S)'].astype(int)
    df['成交價格'] = df['成交價格'].astype(float)

    # 計算 OHLC
    df_ohlc = df['成交價格'].resample(interval).agg({'Open':'first', 'High':'max', 'Low':'min', 'Close':'last'})

    # 計算 Volume
    df_volume = df['成交數量(B+S)'].resample(interval).sum()

    # 將兩個 DataFrame 合併
    df_kline = pd.concat([df_ohlc, df_volume], axis=1)
    df_kline = df_kline.rename(columns={'成交數量(B+S)': 'Volume'})  # 將 '成交數量(B+S)' 重新命名為 'Volume'

    return df_kline

def fetch_time(datetime:datetime, time_start:time=time(8,45), time_end:time=time(13,45), interval:str="5min", symbol:str="MTX"):
    year = datetime.strftime('%Y')
    month = datetime.strftime('%m')
    day = datetime.strftime('%d')

    path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'taifex', f'Daily_{year}_{month}_{day}.csv')
    
    # 讀取檔案，如果當天的資料不存在，則讀取下一個月的資料(轉倉後的資料)
    df = read_taifex(path, symbol, year + month)
    if df.empty:
        # print("test", (datetime + relativedelta(months=1)).strftime('%Y%m'))
        df = read_taifex(path, symbol, (datetime + relativedelta(months=1)).strftime('%Y%m'))
    if df.empty:
        return None
    df_kline = to_kline(interval, df)

    # 轉成固定時間區間
    # df_kline.index = pd.to_datetime(df_kline.index)
    iday = df_kline[df_kline.index.date == datetime.date()]
    iday = iday[iday.index.time >= time_start]
    iday = iday[iday.index.time < time_end]

    return iday

def draw_kline(iday:pd.DataFrame):
    '''
        繪製 K 線圖
        參數:
            iday : pd.DataFrame : 股票資料
        回傳:
            kline_path : str : 圖片檔案路徑
    '''

    mc = mpf.make_marketcolors(up='r',down='g',inherit=True)
    s  = mpf.make_mpf_style(base_mpf_style='tradingview',marketcolors=mc)
    if int(iday.shape[0]) == 0:
        return None, None
    
    # 繪製 K 線圖
    fig, axes = mpf.plot(iday.dropna(), type='candle', style=s,
            ylabel='price',
            volume=True,
            returnfig=True)
    plt.close()

    kline_path = os.path.join("img", "fig.png")
    fig.savefig(kline_path)

    return kline_path

In [40]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain.schema.messages import HumanMessage
from langchain.schema.messages import SystemMessage
from langchain.schema.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                convert_system_message_to_human=True,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

In [41]:

# 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage 設定固定回答
# messages = [
#     SystemMessage(content="你是一個厲害的股市操盤手，專長是當日沖消"),
#     HumanMessage(content="請問k線對於市場情緒的概念是什麼?")
# ]

# # 生成第一次對話回應
# res1 = llm.invoke(messages)
# print(res1.content)
# with open('SBP_candidate/kline3.txt', 'w') as f:
#     f.write(res1.content)

In [42]:
# k 線圖

from datetime import datetime, time
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import timedelta
import json

# 創建日期範圍
start_date = "2023-12-13"
end_date = "2023-12-31"
date_range = pd.date_range(start=start_date, end=end_date)

# 創建時間範圍  9:00 ~ 13:00
time_range = [time(hour) for hour in range(9,14)]

# 讀取 k 
with open('SBP_candidate/kline3.txt', 'r', encoding='utf-8') as f:
    content = f.read()

# 對於每一天的每一小時調用 fetch_time
for date in date_range:
    for start_time in time_range:
        end_time = (datetime.combine(date, start_time) + timedelta(hours=1)).time()
        date_str = date.strftime("%Y%m%d")
        start_time_str = start_time.strftime("%H")
        end_time_str = end_time.strftime("%H")
        formatted_str = f"{date_str}_{start_time_str}_{end_time_str}"
        print(f"date: {date_str}, 從：{start_time_str}～{end_time_str}")
        df = fetch_time(date, start_time, end_time, "5min", "MTX")
        print(df)
        if df is None or len(df)<1:
            continue
        # df.to_csv("temp_data/" + formatted_str + ".csv")
        change = df['Close'].iloc[-1]/df['Open'].iloc[0]
        
        path = draw_kline(df)
        if path is None:
            continue


        # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
        messages = [
            SystemMessage(content="你是一個厲害的股市操盤手，專長是當日沖消"),
            HumanMessage(content="k線對於市場情緒的概念是什麼?")
        ]

        # 設定假的第一次對話回應 以減少使用次數
        res1 = AIMessage(content=content  , response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-3bdfe58e-7707-4a1b-8f35-b8953592f5ff-0')

        # 將 AI 的回應添加到消息列表中
        messages.append(res1)
        messages.append(
            HumanMessage(
                content=[
                    {
                        "type": "text",
                        "text": "這是一張小台指的5分k線圖，請用上面的概念，並透過這張圖的具體形態，關鍵價格，成交量的變化趨勢，深入分析接下來一小時的市場情緒狀態，並猜測其隱含意義",
                    },
                    {"type": "image_url", "image_url": path}  # 替換為實際圖像路徑
                ]
            )
        )
        print(messages)
        res2 = llm.invoke(messages)
        print("res2", res2.content)
        
        res3 = llm.invoke(f"請依據這篇文章{res2.content}如果市場情緒好則只能回答 #long、如果不好則只能回答#short、如果不確定則只能回答 #unknown")
        sig_str = res3.content

        
        if sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        print(sig_str, sig)

        output_data = {
            "date": date_str,
            "start_time": start_time_str,
            "end_time":end_time_str,
            "model": model,
            "temperature": temperature,
            "top_p": top_p,
            "seed": seed,
            "response_sig": sig,
            "change": change,
            "sig_str":sig_str,
        }
        json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
        out_path = os.path.join("out_kline_SBP1")
        os.makedirs(out_path, exist_ok=True)
        with open(out_path + "/" + formatted_str + ".json", 'w', encoding='utf-8') as f:
            f.write(json_content)
        


date: 20231213, 從：09～10
Empty DataFrame
Columns: [Open, High, Low, Close, Volume]
Index: []
date: 20231213, 從：10～11
Empty DataFrame
Columns: [Open, High, Low, Close, Volume]
Index: []
date: 20231213, 從：11～12
Empty DataFrame
Columns: [Open, High, Low, Close, Volume]
Index: []
date: 20231213, 從：12～13
Empty DataFrame
Columns: [Open, High, Low, Close, Volume]
Index: []
date: 20231213, 從：13～14
Empty DataFrame
Columns: [Open, High, Low, Close, Volume]
Index: []
date: 20231214, 從：09～10
None
date: 20231214, 從：10～11
None
date: 20231214, 從：11～12
None
date: 20231214, 從：12～13
None
date: 20231214, 從：13～14
None
date: 20231215, 從：09～10
None
date: 20231215, 從：10～11
None
date: 20231215, 從：11～12
None
date: 20231215, 從：12～13
None
date: 20231215, 從：13～14
None
date: 20231216, 從：09～10
None
date: 20231216, 從：10～11
None
date: 20231216, 從：11～12
None
date: 20231216, 從：12～13
None
date: 20231216, 從：13～14
None
date: 20231217, 從：09～10
None
date: 20231217, 從：10～11
None
date: 20231217, 從：11～12
None
date: 20231217, 從：

c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:308: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


res2 好的，根據提供的圖表，我可以分析如下：

**一、市場情緒分析：**

* **09:00 - 09:10：** 開盤後，小台指出現綠色十字星，顯示多空力量勢均力敵，市場情緒觀望。
* **09:10 - 09:20：** 隨後出現長紅柱，成交量也放大，表明買盤力量湧入，市場情緒轉為看漲。
* **09:20 - 09:30：** 繼續出現紅柱，但長度縮短，成交量也有所減少，顯示買盤力量略有減弱，但市場情緒仍偏向看漲。
* **09:30 - 09:40：** 價格出現短暫回落，但成交量並沒有明顯萎縮，顯示賣盤力量出現，但買盤力量仍然較強。
* **09:40 - 09:50：** 再次出現長紅柱，成交量放大，表明買盤力量再度強勢，市場情緒轉為積極。

**二、關鍵價格分析：**

* **17650 點：** 該點位是重要的支撐位，價格在 09:10 突破該點位，表明買盤力量強勁。
* **17700 點：** 該點位是重要的阻力位，價格在 09:40 突破該點位，表明買盤力量強勁。

**三、成交量分析：**

* 成交量在 09:10 和 09:40 兩個時間段出現放大，顯示市場參與者積極，情緒強烈。

**四、接下來一小時的市場情緒狀態：**

* 目前市場情緒偏向看漲，買盤力量仍然強勢。
* 價格有可能繼續向上突破，但要注意 17700 點的阻力位。
* 觀察成交量變化，如果成交量繼續放大，則市場情緒可能更加看漲，反之則可能出現調整。

**五、隱含意義猜測：**

* 從圖表可以看出，小台指的價格在過去的一個小時內持續走高，這可能是受到一些利好消息的影響。
* 觀察新聞資訊，可以尋找可能影響市場情緒的事件，例如：
    * 經濟數據公布
    * 政策利好消息
    * 國際市場走勢
    * 資金流向等

**總結：**

總體而言，小台指目前市場情緒偏向看漲，但需要密切關注成交量和價格的變化，以判断市场是否会继续上涨或出现调整。

**請注意：**

* 以上分析僅供參考，不構成任何投資建議。
* 交易有風險，投資需謹慎。
* 建議您在進行交易前，做好充分的功課，并结合其他分析工具进行综合判断。


c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:308: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


#long 
 1
date: 20231226, 從：10～11
                        Open     High      Low    Close  Volume
Date                                                           
2023-12-26 10:00:00  17702.0  17715.0  17701.0  17703.0    5748
2023-12-26 10:05:00  17704.0  17707.0  17689.0  17691.0    4410
2023-12-26 10:10:00  17692.0  17696.0  17687.0  17694.0    2898
2023-12-26 10:15:00  17692.0  17697.0  17687.0  17689.0    2016
2023-12-26 10:20:00  17690.0  17692.0  17685.0  17691.0    1988
2023-12-26 10:25:00  17692.0  17703.0  17691.0  17700.0    2630
2023-12-26 10:30:00  17701.0  17702.0  17696.0  17696.0    1070
2023-12-26 10:35:00  17697.0  17702.0  17695.0  17698.0    1008
2023-12-26 10:40:00  17699.0  17702.0  17696.0  17696.0     712
2023-12-26 10:45:00  17697.0  17702.0  17696.0  17699.0     942
2023-12-26 10:50:00  17700.0  17704.0  17696.0  17701.0    1094
2023-12-26 10:55:00  17701.0  17703.0  17700.0  17701.0     470
[SystemMessage(content='你是一個厲害的股市操盤手，專長是當日沖消'), HumanMessage(content='

...

c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:308: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


res2 ## 小台指5分K線圖分析

這張圖顯示了小台指在過去一小時的價格走勢，讓我們根據K線圖形、關鍵價格、成交量變化趨勢來深入分析接下來一小時的市場情緒狀態。

**K線圖形:**

* 12:00-12:20: 以綠色長K線為主，表示買盤力量較強，市場情緒看漲。
* 12:20-12:40: 以綠色短K線和紅色長K線交替出現，表示多空力量勢均力敵，市場情緒猶豫不決。
* 12:40-12:50: 以紅色長K線為主，表示賣盤力量較強，市場情緒看跌。

**關鍵價格:**

* 17730: 這個價格區間是12:00-12:20的最高點，也是12:20-12:40的阻力位。
* 17722: 這個價格區間是12:40-12:50的最低點，也是12:20-12:40的支撐位。

**成交量變化趨勢:**

* 12:00-12:10: 成交量很大，表示市場參與者積極，情緒強烈。
* 12:10-12:30: 成交量逐漸減少，表示市場參與者觀望，情緒冷淡。
* 12:30-12:50: 成交量再次放大，表示市場參與者情緒再次活躍。

**接下來一小時的市場情緒狀態:**

* **短期來看:** 12:40-12:50出現紅色長K線，成交量放大，表明賣盤力量較強，市場情緒偏向看跌。
* **中期來看:** 12:20-12:40的猶豫不決狀態，顯示市場可能在尋找新的方向，而 17730 附近的阻力位可能會阻礙上漲。
* **長期來看:** 12:00-12:20的看漲狀態尚未完全消退，17722 附近的支撐位可能提供一定的底部支撐。

**隱含意義:**

* **市場可能正在經歷短期修正:** 12:40-12:50的看跌走勢，可能暗示市場正在進行短期修正。
* **投資者需要觀察關鍵價格:** 17730 和 17722 附近的價格區間，將決定市場接下來的走勢。
* **成交量變化暗示交易策略:** 成交量放大的時候，表明市場情緒活躍，適合進行短線交易；成交量萎縮的時候，表明市場情緒冷淡，適合觀望。

**結論:**

接下來一小時的市場情緒可能偏向看跌，但市場尚未完全失去上漲動力。投資者需要密切觀察價格走勢和成交量變化，才能做出明智的交易決策。

**注意事項:**

以上分析僅供參考，不構成投資建議。投資者應自行進行分析和判斷，並承擔所有風險。



c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:308: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


#short 
 -1
date: 20231226, 從：13～14
                        Open     High      Low    Close  Volume
Date                                                           
2023-12-26 13:00:00  17727.0  17735.0  17727.0  17735.0     960
2023-12-26 13:05:00  17735.0  17736.0  17731.0  17731.0    1322
2023-12-26 13:10:00  17731.0  17734.0  17730.0  17732.0     646
2023-12-26 13:15:00  17731.0  17733.0  17729.0  17731.0     664
2023-12-26 13:20:00  17729.0  17732.0  17727.0  17728.0    1312
2023-12-26 13:25:00  17728.0  17731.0  17725.0  17730.0    1768
2023-12-26 13:30:00  17731.0  17735.0  17726.0  17734.0    3208
2023-12-26 13:35:00  17734.0  17735.0  17729.0  17732.0    2422
2023-12-26 13:40:00  17731.0  17736.0  17730.0  17734.0    2846
2023-12-26 13:45:00  17734.0  17734.0  17733.0  17733.0       4
[SystemMessage(content='你是一個厲害的股市操盤手，專長是當日沖消'), HumanMessage(content='k線對於市場情緒的概念是什麼?'), AIMessage(content='作為一個厲害的當日沖消交易員，我深知K線圖對於市場情緒的意義重大。K線圖不僅僅是價格變化的記錄，它更是市場情緒的寫照，是交易者心理的晴雨表。\n\n**K線圖如何反映市場情緒？

c:\Users\user\Desktop\LLM-Predict-Stock\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:308: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


In [15]:
import pandas as pd
import json
import os

# 計算準確率和平均報酬率
gain = []
loss = []

tp = 0
fp = 0
tn = 0
fn = 0

max_gain = 0
max_loss = 0

files = os.listdir("out_kline_SBP")
# print(len(files))
files = sorted(files)

for idx, file in enumerate(files):
    if file.split("_")[1] != "09" and file.split("_")[1] != "10" and file.split("_")[1] != "11":
        continue
    # print(file)
    with open("out_kline_SBP/" + file, "r", encoding='utf-8') as f:
        data = json.load(f)
        
        sig_str = data["sig_str"]

        sig = 0
        if sig_str.find("unknown") > 0:
            sig = 0
        elif sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        # print(sig_str, sig)
        # sig = data["response_sig"]

        # print(data["response_sig"], data["sig_str"])

    if idx+1 == len(files):
        break
    # print(files[idx+1])
    with open("out_kline_SBP/" + files[idx+1], "r", encoding='utf-8') as f:
        data = json.load(f)
        rtn = data["change"]
        rtn = float(rtn) - 1
        # print(rtn)
    # print()
    
    if pd.isna(rtn):
        rtn = 0

    if sig > 0:
        if rtn > 0:
            tp += 1
            gain.append(rtn)
            if rtn > max_gain:
                max_gain = rtn

        else:
            fp += 1
            loss.append(rtn)
            if rtn < max_loss:
                max_loss = rtn


    elif sig == -1: 
        if rtn < 0:
            tn += 1
            gain.append(-rtn)
            if -rtn > max_gain:
                max_gain = -rtn

        else:
            fn += 1
            loss.append(-rtn)
            if -rtn < max_loss:
                max_loss = -rtn


print(max_gain, max_loss, tp+fp+tn+fn)
print(tp, fp, tn, fn)

accuracy = (tp + tn) / (tp + fp + tn + fn)
print(f'Accuracy: {accuracy}')
print(f'Average gain: {sum(gain) / len(gain)}')
print(f'Average loss: {sum(loss) / len(loss)}')
print(f'expected value: {(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')



0.009146877748460902 -0.007664946645959625 216
37 40 82 57
Accuracy: 0.5509259259259259
Average gain: 0.0016047986292615174
Average loss: -0.0016044593616085846
expected value: 0.00016360406854670306


9. acc=0.546 ev=0.00022259264938792405
10. acc=0.628 ev=0.00027859863256242444
11. acc=0.4788732394366197 ev= -1.2082734892621865e-05
12. acc=0.5319148936170213 ev=0.0001631474214469264